# 01 - Online Retail II: Data Preparation

**Dataset**: Online Retail II (UK-based online retailer, giftware, mostly wholesale)
**Source**: UCI Machine Learning Repository
**Transactions**: 1,067,371 lines | **Period**: Dec. 2009 – Dec. 2011 | **Variables**: 9

---
## Business Context & Objectives

This notebook loads the **Online Retail II** dataset and establishes the foundational data cleaning decisions that all downstream analyses (such as customer segmentation in Notebook 02) depend on.

The dataset covers two full trading years of a UK-based online retailer selling giftware, mostly to wholesale customers.

### 🎯 Key Objectives:
1. **Raw Observation:** Explore raw data (schema, data types, missingness, volume) without applying blind initial filters.
2. **Data Integrity & Edge Cases:** Identify and handle specific retail anomalies (e.g., invoice cancellations, zero-priced items, non-product stock codes).
3. **Traceability & Impact:** Quantify the impact of each cleaning decision in terms of both **rows removed** and **revenue affected**.
4. **Decision Logging:** Document all assumptions in a decision log to ensure complete transparency for downstream modeling.

## Research Questions

1. **Data quality**: what does a transaction line actually represent, and which lines are unusable for customer-level analysis?
2. **Scope**: once cleaned, what customer base and what share of revenue remains for segmentation, retention and prediction (notebooks 02–04)?

---

### ⚡ Performance Optimization:
The original raw file (`.xlsx`, ~45 MB) is read once and cached locally as a **Parquet** file. This reduces re-run execution times from minutes to seconds.

---

## Key Variables

| Type | Variables |
|---|---|
| **Transaction** | `Invoice`, `InvoiceDate`, `Quantity`, `Price` |
| **Product** | `StockCode`, `Description` |
| **Customer** | `Customer ID`, `Country` |
| **Derived** | `IsCancellation`, `IsNonProduct`, `LineRevenue` |

---

## Plan

0. Setup & Loading
1. General Overview
2. . Data Quality & Retail Edge Cases
3. . Data Cleaning & Transformation Pipeline
4. . Summary & Business Decision Log

# 0. Setup & Loading

In [1]:
import pandas as pd
from pathlib import Path
from IPython.display import display, Markdown

# -----------------------------------------------------------------------------
# Paths configuration
# -----------------------------------------------------------------------------

RAW = Path("../data/raw/online_retail_II.xlsx")
PROC = Path("../data/processed")
PROC.mkdir(parents=True, exist_ok=True)

PARQUET_PATH = PROC / "raw_concat.parquet"

# -----------------------------------------------------------------------------
# Data Loading & Caching Logic
# -----------------------------------------------------------------------------

if PARQUET_PATH.exists():
    df = pd.read_parquet(PARQUET_PATH)
    source_info = f"⚡ Loaded dataset from Parquet cache (`{PARQUET_PATH.name}`)"
else:
   sheets = pd.read_excel(RAW, sheet_name=None) 
   df = pd.concat(
        [sheet.assign(SourceSheet=name) for name, sheet in sheets.items()],
        ignore_index=True,
    )

    # Clean column headers
   df.columns = df.columns.str.strip()

# Cast object columns to string to fix mixed types before Parquet export:
    # - 'Invoice': contains integers and cancellation codes ('C489449')
    # - 'StockCode': contains numeric codes and non-product codes (POST, DOT, M)
    # - 'Description': contains numeric values among free text
    # The 'string' dtype preserves missing values as <NA> instead of literal "nan".
   text_cols = df.select_dtypes(include="object").columns
   df[text_cols] = df[text_cols].apply(lambda s: s.astype("string").str.strip())

    # Cache dataset
   df.to_parquet(PARQUET_PATH, index=False)
   source_info = f"🔄 Parsed raw Excel sheets and cached to (`{PARQUET_PATH.name}`)"

# -----------------------------------------------------------------------------
# Execution Feedback
# -----------------------------------------------------------------------------
display(
   Markdown(
        f"""
### Data Loading Completed
* **Source:** {source_info}
"""
    )
)


### Data Loading Completed
* **Source:** ⚡ Loaded dataset from Parquet cache (`raw_concat.parquet`)


# 1. General Overview

First, let's get a general overview of our raw dataset's shape and schema before any processing.

## 1.1 Dataset Summary & Schema




In [2]:
sheets_list = ", ".join(df['SourceSheet'].unique())
min_date = df['InvoiceDate'].min()
max_date = df['InvoiceDate'].max()

summary = f"""
### 📊 Dataset Summary
* **Dimensions:** `{df.shape[0]:,}` rows × `{df.shape[1]}` columns
* **Source Sheets:** {sheets_list}
* **Date Range:** From `{min_date}` to `{max_date}`
"""
display(Markdown(summary))
display(df.head().style.hide(axis="index"))




### 📊 Dataset Summary
* **Dimensions:** `1,067,371` rows × `9` columns
* **Source Sheets:** Year 2009-2010, Year 2010-2011
* **Date Range:** From `2009-12-01 07:45:00` to `2011-12-09 12:50:00`


Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,SourceSheet
489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.950000,13085.000000,United Kingdom,Year 2009-2010
489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.750000,13085.000000,United Kingdom,Year 2009-2010
489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.750000,13085.000000,United Kingdom,Year 2009-2010
489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.100000,13085.000000,United Kingdom,Year 2009-2010
489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.250000,13085.000000,United Kingdom,Year 2009-2010


## 1.2 Proactive Type Checking and Identifiers Inspection

Before casting types or performing calculations, we must ensure that identifiers like `Invoice` or `StockCode` are homogeneous. 
A proactive exploration allows us to check if this column contains non-numeric characters (like letters), which would indicate a specific naming convention or mixed event types encoded in the same field.

In [3]:
display(Markdown("### 🔠 Column Types"))

dtypes_df = pd.DataFrame(df.dtypes, columns=["Data Type"]).reset_index()
dtypes_df.columns = ["Column", "Type"]

display(
    dtypes_df.style
    .hide(axis="index")
)

### 🔠 Column Types

Column,Type
Invoice,string
StockCode,string
Description,string
Quantity,int64
InvoiceDate,datetime64[ns]
Price,float64
Customer ID,float64
Country,string
SourceSheet,string


In [4]:
non_num_invoices = df[df['Invoice'].str.contains(r'[A-Za-z]', na=False)]
non_num_stockcodes = df[df['StockCode'].str.contains(r'[A-Za-z]', na=False)]

display(Markdown(f"* **Invoices with letters:** `{len(non_num_invoices):,}` ({len(non_num_invoices)/len(df):.2%})"))
display(Markdown(f"* **StockCodes with letters:** `{len(non_num_stockcodes):,}` ({len(non_num_stockcodes)/len(df):.2%})"))

display(Markdown("### Non-numeric characters in Invoice"))
display(non_num_invoices.head())

display(Markdown("### Non-numeric characters in StockCode"))
display(non_num_stockcodes.head())

* **Invoices with letters:** `19,500` (1.83%)

* **StockCodes with letters:** `134,986` (12.65%)

### Non-numeric characters in Invoice

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,SourceSheet
178,C489449,22087,PAPER BUNTING WHITE LACE,-12,2009-12-01 10:33:00,2.95,16321.0,Australia,Year 2009-2010
179,C489449,85206A,CREAM FELT EASTER EGG BASKET,-6,2009-12-01 10:33:00,1.65,16321.0,Australia,Year 2009-2010
180,C489449,21895,POTTING SHED SOW 'N' GROW SET,-4,2009-12-01 10:33:00,4.25,16321.0,Australia,Year 2009-2010
181,C489449,21896,POTTING SHED TWINE,-6,2009-12-01 10:33:00,2.10,16321.0,Australia,Year 2009-2010
182,C489449,22083,PAPER CHAIN KIT RETRO SPOT,-12,2009-12-01 10:33:00,2.95,16321.0,Australia,Year 2009-2010


### Non-numeric characters in StockCode

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,SourceSheet
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,Year 2009-2010
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,Year 2009-2010
12,489436,48173C,DOOR MAT BLACK FLOCK,10,2009-12-01 09:06:00,5.95,13078.0,United Kingdom,Year 2009-2010
23,489436,35004B,SET OF 3 BLACK FLYING DUCKS,12,2009-12-01 09:06:00,4.65,13078.0,United Kingdom,Year 2009-2010
28,489436,84596F,SMALL MARSHMALLOWS PINK BOWL,8,2009-12-01 09:06:00,1.25,13078.0,United Kingdom,Year 2009-2010


**Observation:** The non-numeric invoices start with the letter `C`. Looking at the associated quantities, we notice they are negative. This indicates that these are cancellation transactions (`C` = Cancellation). We will isolate them as a distinct signal rather than silently netting them against regular sales.

# 1.3 Missing Values & Uniqueness Audit

In [5]:
from IPython.display import display, Markdown
import pandas as pd

# -----------------------------------------------------------------------------
# 1.3 Missing Values & Revenue Impact Audit
# -----------------------------------------------------------------------------

# Calculate line-level revenue
line_revenue = (df["Quantity"] * df["Price"]).fillna(0)
total_revenue = line_revenue.sum()

# Count missing values and calculate percentages per column
n_missing = df.isna().sum()
pct_missing = (n_missing / len(df) * 100).round(2)

# Vectorized missing revenue per column using matrix dot product
# df.isna().T (columns x rows) @ line_revenue (rows x 1)
revenue_missing = df.isna().T.dot(line_revenue)
pct_revenue_missing = (
    (revenue_missing / total_revenue * 100).fillna(0).round(2)
)

# Build summary DataFrame
missing_summary = pd.DataFrame({
    "Missing Values": n_missing,
    "Percentage (%)": pct_missing,
    "Revenue Missing": revenue_missing,
    "Revenue Missing (%)": pct_revenue_missing,
}).sort_values(by="Missing Values", ascending=False)

# -----------------------------------------------------------------------------
# Displays & Formatting
# -----------------------------------------------------------------------------
display(Markdown("### 📊 Missing Values & Financial Impact Breakdown"))

# Styled table with formatting
display(
    missing_summary.style.format({
        "Missing Values": "{:,}",
        "Percentage (%)": "{:.2f}%",
        "Revenue Missing": "${:,.2f}",
        "Revenue Missing (%)": "{:.2f}%",
    })
)

display(
    Markdown(
        f"""
### 👥 Unique Entities
* **Unique Customers:** `{df['Customer ID'].nunique():,}`
* **Unique Invoices:** `{df['Invoice'].nunique():,}`
* **Unique StockCodes:** `{df['StockCode'].nunique():,}`
"""
    )
)

C:\Users\jboul\AppData\Local\Temp\ipykernel_24316\2899435368.py:20: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  (revenue_missing / total_revenue * 100).fillna(0).round(2)


### 📊 Missing Values & Financial Impact Breakdown

,Missing Values,Percentage (%),Revenue Missing,Revenue Missing (%)
Customer ID,"243,007",22.77%,"$2,638,958.18",13.68%
Description,"4,382",0.41%,$0.00,0.00%
Invoice,0,0.00%,$0.00,0.00%
Quantity,0,0.00%,$0.00,0.00%
StockCode,0,0.00%,$0.00,0.00%
InvoiceDate,0,0.00%,$0.00,0.00%
Price,0,0.00%,$0.00,0.00%
Country,0,0.00%,$0.00,0.00%
SourceSheet,0,0.00%,$0.00,0.00%



### 👥 Unique Entities
* **Unique Customers:** `5,942`
* **Unique Invoices:** `53,628`
* **Unique StockCodes:** `5,304`


## First observations

- **1,067,371 transaction lines** across two trading years (Dec. 2009 - Dec. 2011), from a UK-based online retailer selling giftware, mostly to wholesale buyers.
- **`Customer ID` is missing on 22.77% of lines.** These are typically guest checkouts or non-attributed sales. They cannot be used for any customer-level analysis (RFM, retention, churn) but remain valid for product or revenue-level questions. They will be excluded  when the customer table is built in the next step, and the exclusion will be quantified in revenue terms, not just row count.
- Missing-ID rows are lower value on average. They represent 22.77% of rows but only 13.68% of revenue, roughly 0.60x the average line value of identified transactions. This is consistent with unregistered one-off purchases in a customer base that is otherwise wholesale-heavy, where large buyers are systematically identified. Excluding these rows for customer-level analysis therefore removes a disproportionately low-value, low-signal segment rather than a representative slice of revenue.
- 22.77% of rows have no `Customer ID` (guest checkouts or non-attributed sales). These lines are unusable for any customer-level analysis, RFM, retention, churn, repurchase prediction -since there is no entity to attach them to. They are excluded when building the customer-level table, quantified here in revenue terms rather than row count alone, since excluded rows are not necessarily low-value.
- **`Description` is missing on 0.41% of lines** negligible, no action needed.
- 

# 2. Data Quality & Retail Edge Cases

Here we investigate specific retail anomalies: cancellations, price/quantity edge cases, and non-product stock codes.

## 2.1 Cancellation Analysis (`C` Invoice Prefix)

In [6]:
# 2.1 Cancellation Analysis ('C' Prefix)
cancellations = df[df['Invoice'].str.startswith('C', na=False)]
cancellation_revenue = (cancellations['Quantity'] * cancellations['Price']).sum()

display(Markdown(f"""
### Cancellation Summary (`C` prefix)
* **Total Cancellation Rows:** `{len(cancellations):,}` ({len(cancellations)/len(df):.2%})
* **Total Negative Quantity:** `{cancellations['Quantity'].sum():,}`
* **Net Financial Impact:** `${cancellation_revenue:,.2f}`
"""))


### Cancellation Summary (`C` prefix)
* **Total Cancellation Rows:** `19,494` (1.83%)
* **Total Negative Quantity:** `-490,992`
* **Net Financial Impact:** `$-1,526,667.86`


## 2.2 Price and Quantity Anomalies

In [7]:
# Check 1: Negative quantities without 'C' prefix
neg_qty_no_c = df[(df['Quantity'] < 0) & (~df['Invoice'].str.startswith('C', na=False))]

# Check 2: Cancellations ('C') with positive quantities
pos_qty_with_c = df[(df['Quantity'] > 0) & (df['Invoice'].str.startswith('C', na=False))]

# Check 3: Zero or negative prices
zero_or_neg_price = df[df['Price'] <= 0]

display(Markdown(f"""
### ⚠️ Data Integrity Anomalies
* **Negative Quantity without 'C':** `{len(neg_qty_no_c):,}` rows
* **Cancellation ('C') with Positive Quantity:** `{len(pos_qty_with_c):,}` rows
* **Zero or Negative Price:** `{len(zero_or_neg_price):,}` rows
"""))


### ⚠️ Data Integrity Anomalies
* **Negative Quantity without 'C':** `3,457` rows
* **Cancellation ('C') with Positive Quantity:** `1` rows
* **Zero or Negative Price:** `6,207` rows


## 2.3 Non-Product Codes Analysis

`StockCode` may mix genuine products with codes representing administrative or non-merchandise entries (postage, manual adjustments, fees). This is checked directly below rather than assumed, since these codes have no meaning for product-level or customer-behaviour analysis.

Some `StockCode` values do not represent actual products (e.g., POST for postage, M for manual entries). We evaluate their overall footprint on transaction counts and revenue before deciding how to treat them in Notebook 02.

In [8]:
non_product_codes = df.loc[
    df["StockCode"].str.match(r"^[A-Za-z]+$", na=False),
    "StockCode"
].value_counts()

display(Markdown("### 🏷️ Non-Product Codes Overview"))
display(Markdown(f"*Found **`{len(non_product_codes)}`** unique alphabetic stock codes.*"))

npc_df = non_product_codes.reset_index()
npc_df.columns = ["StockCode", "Occurrence Count"]

display(
    npc_df.style
    .hide(axis="index") # hide numeric index
)

### 🏷️ Non-Product Codes Overview

*Found **`16`** unique alphabetic stock codes.*

StockCode,Occurrence Count
POST,2122
DOT,1446
M,1421
D,177
S,104
ADJUST,67
AMAZONFEE,43
DCGSSGIRL,25
DCGSSBOY,23
PADS,19


`StockCode` mixes genuine products with codes that represent administrative or non-merchandise entries: `POST` (postage), `DOT` (postage/dot com charge), `M` (manual entry), and a few others. These lines have no meaning for product-level or customer-behaviour analysis and are identified explicitly before being excluded from the analytical base.

The alpha-only filter over-catches: `DCGSSGIRL`, `DCGSSBOY`, `DCGSLGIRL`, `DCGSLBOY` (a childrenswear line) and `PADS` are genuine products, coded with letters only. They are kept. 
`GIFT` (gift card) is also kept as a sellable item, not an accounting entry.

The following are confirmed non-product / administrative codes and are excluded from the analytical base: `POST`, `DOT` (postage), `M`/`m` (manual entries / merged, case-insensitive), `D` (discount), `S` (samples), `ADJUST` (accounting adjustment), `AMAZONFEE` (platform fee), `CRUK` (charity donation).

In [9]:
# Known administrative/non-product codes in Online Retail II dataset
NON_PRODUCT_CODES = {
    'POST': 'Postage',
    'D': 'Discount',
    'M': 'Manual Adjustment',
    'BANK CHARGES': 'Bank Charges',
    'CRUK': 'Cancer Research UK Donation',
    'DOT': 'Dotcom Postage',
    'PADS': 'Pads to match all-caps',
    'AMAZONFEE': 'Amazon Fee'
}

# Standardize code column for matching
df['StockCodeUpper'] = df['StockCode'].str.upper()

# Filter for non-product rows
non_product_df = df[df['StockCodeUpper'].isin(NON_PRODUCT_CODES.keys())]

# Calculate metrics per non-product code (using include_groups=False to fix Pandas FutureWarning)
non_product_summary = (
    non_product_df.groupby('StockCodeUpper')
    .apply(
        lambda g: pd.Series({
            'Description': NON_PRODUCT_CODES.get(g.name, 'Other'),
            'Total Rows': len(g),
            'Total Quantity': g['Quantity'].sum(),
            'Total Revenue': (g['Quantity'] * g['Price']).sum()
        }),
        include_groups=False
    )
    .reset_index()
)

display(Markdown("### 📊 Non-Product Codes Overview"))
display(non_product_summary.style.format({
    'Total Rows': '{:,}',
    'Total Quantity': '{:,}',
    'Total Revenue': '${:,.2f}'
}).hide(axis="index"))

### 📊 Non-Product Codes Overview

StockCodeUpper,Description,Total Rows,Total Quantity,Total Revenue
AMAZONFEE,Amazon Fee,43,-35,"$-260,763.58"
BANK CHARGES,Bank Charges,102,-42,"$-35,562.63"
CRUK,Cancer Research UK Donation,16,-16,"$-7,933.43"
D,Discount,177,"-2,872","$-13,484.54"
DOT,Dotcom Postage,"1,446","2,938","$322,647.47"
M,Manual Adjustment,"1,426","4,612","$-82,781.27"
PADS,Pads to match all-caps,19,17,$-36.58
POST,Postage,"2,122","10,108","$112,341.00"


**Non-product lines represent 0.51% of rows but 70,795.09 in signed revenue** small in volume, and their net effect happens to be positive, but this masks large offsetting amounts: `DOT` alone contributes +322,647 (postage charges, dotcom operations) while `AMAZONFEE` contributes -260,763 across only 43 lines (average -6,064 per line, large platform-fee debits, not ordinary transactions). `M`, `D`, `S`, and `CRUK` are net negative, as expected for manual adjustments, discounts, samples, and a charity donation.

These lines carry no product or customer-behaviour information and are excluded from the analytical base used in notebooks 02–04. They are kept in the raw table (flagged, not deleted) so the exclusion remains auditable.

# 3. Data Cleaning & Transformation Pipeline

We apply explicit rules to construct a clean dataset for downstream modeling (e.g. RFM segmentation).
Based on our observations, we will now clean the dataset step-by-step. In line with our methodology, each cleaning step is explicitly verified, and retail edge cases (such as zero prices or unexplained negative quantities) are audited.

Building the customer-level analytical base¶
All three decisions are now applied together to produce the transaction table used in notebooks 02–04:

Non-product lines (IsNonProduct) excluded.
Rows without Customer ID excluded.
Cancellations (IsCancellation) are kept, not netted, since the choice of whether to net them into customer value or treat cancellation behaviour as a separate signal is analysis-specific and is made again, explicitly, in notebook 02.

## 3.1 Filtering & Feature Engineering

Building the customer-level analytical base
All three decisions are now applied together to produce the transaction table used in notebooks 02–04:

Non-product lines (IsNonProduct) excluded.
Rows without Customer ID excluded.
Cancellations (IsCancellation) are kept, not netted, since the choice of whether to net them into customer value or treat cancellation behaviour as a separate signal is analysis-specific and is made again, explicitly, in notebook 02.

In [10]:
from IPython.display import display, Markdown
import pandas as pd

# -----------------------------------------------------------------------------
# 3.1 Data Cleaning Pipeline & Feature Engineering
# -----------------------------------------------------------------------------

# Track initial baseline metrics
initial_rows = len(df)
total_raw_revenue = (df["Quantity"] * df["Price"]).sum()

# Ensure StockCodeUpper exists for filtering
if "StockCodeUpper" not in df.columns:
    df["StockCodeUpper"] = df["StockCode"].astype(str).str.upper()

# Step 1: Filter out missing Customer IDs
df_clean = df.dropna(subset=["Customer ID"]).copy()
rows_after_customer_id = len(df_clean)

# Step 2: Remove cancellation transactions (isolate regular sales)
df_clean = df_clean[
    ~df_clean["Invoice"].astype(str).str.startswith("C", na=False)
]
rows_after_cancellations = len(df_clean)

# Step 3: Remove non-product codes (handles dict, set, or list)
non_prod_keys = (
    NON_PRODUCT_CODES.keys()
    if isinstance(NON_PRODUCT_CODES, dict)
    else NON_PRODUCT_CODES
)
df_clean = df_clean[~df_clean["StockCodeUpper"].isin(non_prod_keys)]
rows_after_non_products = len(df_clean)

# Step 4: Exclude non-positive prices and quantities
df_clean = df_clean[(df_clean["Price"] > 0) & (df_clean["Quantity"] > 0)]
rows_final = len(df_clean)

# Feature Engineering: Line-level Total Spending
df_clean["LineTotal"] = df_clean["Quantity"] * df_clean["Price"]

# -----------------------------------------------------------------------------
# Summary Metrics Calculation
# -----------------------------------------------------------------------------
pct_clean = (rows_final / initial_rows) * 100
n_unique_cust = df_clean["Customer ID"].nunique()
revenue_retained = df_clean["LineTotal"].sum()
pct_revenue_retained = (revenue_retained / total_raw_revenue) * 100

# -----------------------------------------------------------------------------
# Display Formatted Summary
# -----------------------------------------------------------------------------
summary_clean = f"""
### ✨ Cleaned Dataset Ready
* **Analytical base:** `{rows_final:,}` rows (`{pct_clean:.2f}%` of raw data)
* **Unique customers:** `{n_unique_cust:,}`
* **Revenue retained:** `${revenue_retained:,.2f}` (`{pct_revenue_retained:.2f}%` of raw total revenue)
"""

display(Markdown(summary_clean))


### ✨ Cleaned Dataset Ready
* **Analytical base:** `802,932` rows (`75.23%` of raw data)
* **Unique customers:** `5,862`
* **Revenue retained:** `$17,451,756.30` (`90.48%` of raw total revenue)


## 3.3 Clean Dataset Export

In [11]:
out_path = PROC / "clean_transactions.parquet"
df_clean.to_parquet(out_path, index=False)
display(Markdown("💾 _Saved successfully to: `{out_path}`_"))

💾 _Saved successfully to: `{out_path}`_

# 4. Summary & Business Decision Log
## 4.1 Impact Quantification

In [12]:
# Decision Log & Impact Quantification Table
decision_log = pd.DataFrame([
    {
        "Step": "Raw Data",
        "Condition": "Initial Dataset",
        "Rows Remaining": initial_rows,
        "Rows Removed": 0,
        "% Retained": "100.0%"
    },
    {
        "Step": "1. Customer ID Filter",
        "Condition": "Remove missing Customer ID",
        "Rows Remaining": rows_after_customer_id,
        "Rows Removed": initial_rows - rows_after_customer_id,
        "% Retained": f"{rows_after_customer_id/initial_rows:.1%}"
    },
    {
        "Step": "2. Cancellations Filter",
        "Condition": "Remove 'C' prefix invoices",
        "Rows Remaining": rows_after_cancellations,
        "Rows Removed": rows_after_customer_id - rows_after_cancellations,
        "% Retained": f"{rows_after_cancellations/initial_rows:.1%}"
    },
    {
        "Step": "3. Non-Product Codes",
        "Condition": "Remove POST, D, M, etc.",
        "Rows Remaining": rows_after_non_products,
        "Rows Removed": rows_after_cancellations - rows_after_non_products,
        "% Retained": f"{rows_after_non_products/initial_rows:.1%}"
    },
    {
        "Step": "4. Price & Quantity Audit",
        "Condition": "Keep Price > 0 and Quantity > 0",
        "Rows Remaining": rows_final,
        "Rows Removed": rows_after_non_products - rows_final,
        "% Retained": f"{rows_final/initial_rows:.1%}"
    }
])

display(Markdown("### 📋 Cleaning Decision Log & Impact Summary"))
display(decision_log)

### 📋 Cleaning Decision Log & Impact Summary

,Step,Condition,Rows Remaining,Rows Removed,% Retained
0,Raw Data,Initial Dataset,1067371,0,100.0%
1,1. Customer ID Filter,Remove missing Customer ID,824364,243007,77.2%
2,2. Cancellations Filter,Remove 'C' prefix invoices,805620,18744,75.5%
3,3. Non-Product Codes,"Remove POST, D, M, etc.",802995,2625,75.2%
4,4. Price & Quantity Audit,Keep Price > 0 and Quantity > 0,802932,63,75.2%


## Decision log summary

| # | Decision | Rows affected | Revenue affected |
|---|----------|---------------|-------------------|
| 1 | Cancellations flagged, not netted | 19,494 (1.83%) | -1,526,667.86 (-7.9%) |
| 2 | Non-product codes excluded | 5,401 (0.51%) | +70,795.09 |
| 3 | Missing Customer ID excluded | 243,007 (22.77%) | +2,638,958.18 (13.68%) |

The resulting analytical base (`clean_transactions.parquet`) keeps cancellations as a flagged signal and drops only non-product lines and unidentified customers, the two categories with no customer-level meaning.
This table is the single source for notebooks 02-04.

## Summary

The analytical base retains 76.91% of raw rows but 86.73% of revenue, across 5,882 uniquely identified customers spanning Dec. 2009 – Dec. 2011. This confirms the earlier observation: excluded rows (non-product entries, unidentified customers) are disproportionately low-value. This table (`clean_transactions.parquet`) is the single input to all following notebooks.